# Hands-on: Bring observations into an Anemoi model

So far our models learnt from **reanalysis** (ERA5): complete, gridded fields. In this session we take the deterministic model of session 5 and give it a **second dataset**, the way we would add observations: with its own nodes in the graph and its own encoder or decoder.

We use the **multi-dataset** support of anemoi: besides the ERA5 dataset `data` (O48), the model gets a second dataset, `obs`. Depending on where `obs` enters the model, we get two setups:

| | Setup | `obs` is | What it is good for |
|---|---|---|---|
| **1** | **Decode to observations** (main part of this session) | a *target*: a new decoder predicts it | predict the observed quantities wherever we want, including variables the model never had |
| **2** | **Encode observations** | an *input*: a new encoder reads it | correct the forecast with the latest observations: a first step towards data assimilation |

```
   1: decode to obs                              2: encode obs
data ─enc─► hidden ─proc─► dec ─► data         data ─enc─┐
                            └─dec─► obs        obs  ─enc─┴(+)─► hidden ─proc─► dec ─► data
```

**Our stand-in for observations.** To keep things simple, `obs` is **ERA5 on the O96 grid** (four times as many points as O48), with a few variables the pretrained model never had (wind, humidity). It is a complete field: every grid point is "observed" at every time. Real observations are sparse and irregular; we discuss what changes for them along the way.

**anemoi-core `main` vs. the next release.** This notebook runs on the current `main` branch of anemoi-core, which only supports **gridded** datasets: every dataset is a field on a fixed set of points. The next release ("richer batches", section 8) adds **tabular** observations: each sample holds the observations there are, with their own positions and times. Everything we set up here (datasets, graph, encoders and decoders, losses, fine-tuning) carries over.

**Resources**
- [anemoi-training: multiple datasets](https://anemoi.readthedocs.io/projects/training/en/latest/)
- [anemoi-models: multi-dataset model configs](https://anemoi.readthedocs.io/projects/models/en/latest/)
- [Alexe et al. 2024, GraphDOP: Towards skilful data-driven medium-range weather forecasts learnt and initialised directly from observations](https://arxiv.org/abs/2412.15687)

## 0. Setup

This session needs anemoi-core `main` (anemoi-training ≥ 0.10, anemoi-models ≥ 0.13): many config keys changed since the version of session 5, e.g. `task:`, `training.optimization` and `training.checkpoint`. Use the kernel of this session, not the one of session 5.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

from importlib.metadata import version
from pathlib import Path

import cartopy.crs as ccrs
import matplotlib.pyplot as plt
import mlflow
import numpy as np
import pandas as pd
import torch
from omegaconf import OmegaConf

from helpers import (config_diff, latest_run, load_config, open_datasets, plot_metrics, predict, run_training,
                     scatter_map, target)

for package in ["anemoi-training", "anemoi-models", "anemoi-graphs", "anemoi-datasets"]:
    print(f"{package:16} {version(package)}")
pd.set_option("display.float_format", "{:.3g}".format)  # 3 significant digits in tables
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none, training will be very slow")

In [ ]:
# The ERA5 toy dataset of session 5 (O48, 6-hourly, 2010-2022)
DATASET = "/datasets/aifs-ea-an-oper-0001-mars-o48-2010-2022-6h-v1-toy.zarr"
# The second dataset, our stand-in for gridded observations: ERA5 on the O96 grid (the toy dataset of session 6)
OBS_DATASET = "/datasets/aifs-ea-an-oper-0001-mars-o96-6h-v8-profiling-toy.zarr"
# Where the trainings write their output
OUTPUT_ROOT = Path.home() / "anemoi-output"
# The model we fine-tune: the run of configs/pretrain_o48.yaml (the deterministic model of session 5, trained
# with anemoi-core main). Its *training* checkpoint (last.ckpt) is in <OUTPUT_ROOT>/checkpoint/<run id>/.
# If it does not exist, section 5.0 trains it.
PRETRAINED_RUN_ID = "4438386d8b894aa59beb57bb1c7e0b85"
PRETRAINED = OUTPUT_ROOT / "checkpoint" / PRETRAINED_RUN_ID / "last.ckpt"

OVERRIDES = [f"system.input.dataset={DATASET}", f"system.input.obs_dataset={OBS_DATASET}", f"system.output.root={OUTPUT_ROOT}/"]

## 1. The second dataset

The second dataset is an ordinary anemoi dataset (a Zarr store), so we open it with `open_dataset` like the ERA5 one.

In [ ]:
from anemoi.datasets import open_dataset

era5 = open_dataset(DATASET)
obs_all = open_dataset(OBS_DATASET)
print(f"ERA5: {era5.shape} on {era5.resolution}, {era5.dates[0]} to {era5.dates[-1]}, every {era5.frequency}")
print(f"obs:  {obs_all.shape} on {obs_all.resolution}, {obs_all.dates[0]} to {obs_all.dates[-1]}, every {obs_all.frequency}")
print("ERA5 variables:", era5.variables)
print("obs variables: ", obs_all.variables)

The O96 dataset has many more variables than the toy dataset of session 5, among them wind `u`, `v` and humidity `q` on pressure levels, which the pretrained model never had. We use a few of them, plus the variables that describe each grid point and time (the *forcings*: coordinates, time, insolation, land-sea mask, orography). We select them with `select` when opening the dataset; the same option goes into the training config:

In [ ]:
OBSERVED = ["t_850", "t_500", "z_500", "u_500", "v_500", "q_700"]
FORCINGS = ["cos_latitude", "cos_longitude", "sin_latitude", "sin_longitude", "cos_julian_day", "cos_local_time",
            "sin_julian_day", "sin_local_time", "insolation", "lsm", "z"]
obs = open_dataset(OBS_DATASET, select=FORCINGS + OBSERVED)
print(obs.variables)
print("in ERA5 O48 too:", [v for v in OBSERVED if v in era5.variables])
print("new:            ", [v for v in OBSERVED if v not in era5.variables])

In [ ]:
date = obs.dates[len(obs.dates) // 2]
fig, axes = plt.subplots(1, 2, figsize=(16, 4.5), subplot_kw={"projection": ccrs.PlateCarree()})
for ax, ds, name in zip(axes, [era5, obs], ["ERA5 O48 (data)", "ERA5 O96 (obs)"]):
    values = ds[int(np.where(ds.dates == date)[0][0])][ds.name_to_index["t_850"], 0]
    im = scatter_map(ax, ds.latitudes, ds.longitudes, values, title=f"t_850 of {name}, {date} ({len(values)} points)",
                     s=2 if ds is obs else 6, cmap="RdYlBu_r", vmin=240, vmax=300)
plt.colorbar(im, ax=axes, shrink=0.8, label="K")

**Real observations** would look very different: at a given time, almost every grid point would be empty (NaN), e.g. radiosondes are launched at a few hundred places, mostly over land in the northern hemisphere and mostly at 00 and 12 UTC. Two things then change:
- On `main` a dataset must not contain NaNs after its processors, so the gaps have to be **filled in** (an imputer, e.g. with the mean), and the filled-in values must be **left out of the loss** (a mask, the `nan_mask_weights` scaler, taken from the gaps of the *target* time).
- The model sees the observations only where they are, so it learns little about places without observations.

With tabular observations (section 8), no filling-in and no mask are needed. Here, with complete fields, we can focus on the model.

### Normalisation
Like any other variable, the variables of `obs` are normalised with the statistics of their own dataset:

In [ ]:
variables = ["t_850", "z_500"]
pd.DataFrame({
    "obs mean": [obs.statistics["mean"][obs.name_to_index[v]] for v in variables],
    "ERA5 O48 mean": [era5.statistics["mean"][era5.name_to_index[v]] for v in variables],
    "obs stdev": [obs.statistics["stdev"][obs.name_to_index[v]] for v in variables],
    "ERA5 O48 stdev": [era5.statistics["stdev"][era5.name_to_index[v]] for v in variables],
}, index=variables)

**Question:** the two datasets hold the same reanalysis, so the statistics are nearly the same. Over which period were they computed? Real observations would have quite different statistics: radiosondes are mostly launched over land in the northern hemisphere. How would that show?

### Matching the dates of the datasets

In every training sample, the model sees ERA5 and `obs` **at the same times**. On anemoi-core `main`, the samples of the datasets are matched by their **position** in the selected period, not by their date: nothing warns us if the dates differ! So every dataset must have the same frequency and exactly the same dates.

Our O96 dataset is small: it only holds a few days at the end of 2020 (for training) and at the start of 2021 (for validation). We therefore select exactly these dates, for **both** datasets:

In [ ]:
def period(dates, year):
    """First and last date of `year` in `dates`, after checking that they follow each other every 6 hours."""
    d = dates[dates.astype("datetime64[Y]") == np.datetime64(str(year), "Y")]
    assert len(d) and np.all(np.diff(d) == np.timedelta64(6, "h")), f"the dates of {year} are not 6-hourly without gaps"
    return str(d[0]), str(d[-1])


train_period, valid_period = period(obs.dates, 2020), period(obs.dates, 2021)
print("training:  ", train_period)
print("validation:", valid_period)
assert all(np.datetime64(d) in era5.dates for d in train_period + valid_period)

# The configs (configs/obs/era5_o96.yaml) take the periods of `data` from those of `obs`
DATES = [f"dataloader.training.datasets.obs.start={train_period[0]}", f"dataloader.training.datasets.obs.end={train_period[1]}",
         f"dataloader.validation.datasets.obs.start={valid_period[0]}", f"dataloader.validation.datasets.obs.end={valid_period[1]}"]

# sample k of each dataset, as the training sees it: without care, and with the periods above
for k in range(3):
    print(f"sample {k}: ERA5 {era5.dates[k]} obs {obs.dates[k]}   |   "
          f"ERA5 {open_dataset(DATASET, start=train_period[0], end=train_period[1]).dates[k]} "
          f"obs {open_dataset(OBS_DATASET, start=train_period[0], end=train_period[1]).dates[k]}")

**Question:** how many training samples do we get from this period, with 2 input times and 1 target time? What does that mean for how long we can fine-tune before the model just learns these few days by heart?

## 2. A model with two datasets: the config

All configs of this session are in `configs/`. Files that are not there (e.g. `model/graphtransformer.yaml`) come from the installed anemoi-training.

| Config | What it is |
|---|---|
| `pretrain_o48.yaml` | the pretrained model: ERA5 only (dataset `data`), the deterministic model of session 5 |
| `finetune_obs_decoder.yaml` | **setup 1**: pretrained model + decoder to `obs` |
| `encode_obs.yaml` | **setup 2**: pretrained model + encoder of `obs` |
| `obs/era5_o96.yaml` | the dataset `obs`: variables, files, dates; used by both setups |
| `graph/pretrain_o48.yaml`, `graph/finetune_obs_decoder.yaml`, `graph/encode_obs.yaml` | the full graph of each case: nodes, edges, attributes |

With several datasets, most sections of the config have one entry per dataset, keyed by the **dataset name** (`data`, `obs`). The name is also the name of the dataset's nodes in the graph.

| Section | Per dataset |
|---|---|
| `data.datasets.<name>` | which variables are forcings or diagnostics, normalisation |
| `dataloader.{training,validation,test}.datasets.<name>` | which zarr, which variables, which dates |
| `graph.nodes.<name>` | the nodes of the dataset (its grid points) |
| `graph.edges` | edges `<name> → hidden` (encoder) and `hidden → <name>` (decoder) |
| `model.encoders.<group>.source_datasets` | which datasets an encoder reads |
| `model.decoders.<group>.target_datasets` | which datasets a decoder predicts |
| `model.residual.datasets.<name>`, `model.output_mask.datasets.<name>` | skip connection (encoded datasets), output mask |
| `training.training_loss.datasets.<name>` | loss of the dataset (no entry: no loss) |
| `training.scalers.datasets.<name>`, `training.metrics.datasets.<name>`, ... | loss weights and validation metrics |

Encoders and decoders are organised in **groups** with a name of your choice (`"0"` in the default model config). The name appears in the names of the model weights, so the checkpoint of the pretrained model has `encoder.0` and `decoder.0`, and our new decoder will be `decoder.obs`.

Here is everything about the dataset `obs`:

In [ ]:
print(Path("configs/obs/era5_o96.yaml").read_text())

The **role of the variables** decides what `obs` is to the model:
- `diagnostic`: output only. The model predicts them but does not get them as input.
- `forcing`: input only.
- all others are `prognostic`: input and output.

In setup 1, the observed variables are **diagnostic**. Together with a decoder and no encoder, `obs` is an *output-only* dataset. Its forcings are still read: the decoder uses them to describe each point it predicts (section 4).

Now let's compare the full configs of the pretrained model and of setup 1. `config_diff` lists every setting that differs:

In [ ]:
FINETUNE = [f"training.checkpoint.source.path={PRETRAINED}"]  # for the configs that start from the pretrained model
pretrain = load_config("pretrain_o48", OVERRIDES)
finetune = load_config("finetune_obs_decoder", OVERRIDES + DATES + FINETUNE)
pd.set_option("display.max_colwidth", 120)
pd.DataFrame(config_diff(pretrain, finetune), columns=["setting", "pretrain_o48", "finetune_obs_decoder"])

## 3. The graph

Each case has its full graph config in `configs/graph/`. The graph of setup 1 is the graph of the pretrained model (`data` and `hidden` nodes and their edges, unchanged so that the pretrained weights fit) plus a third set of nodes, the 40,320 points of the O96 grid, and the edges of the obs decoder: each obs node receives from its 3 nearest hidden nodes (`KNNEdges`).

In [ ]:
print(Path("configs/graph/finetune_obs_decoder.yaml").read_text())

**Question:** compare with `configs/graph/pretrain_o48.yaml` and `configs/graph/encode_obs.yaml` (setup 2). What is the same in all three, and why must it be?

In [ ]:
from anemoi.graphs.create import GraphCreator

graph = GraphCreator(config=finetune.graph).create()
print(graph)

In [ ]:
def lonlat(nodes):
    lat, lon = np.rad2deg(graph[nodes].x.cpu().numpy()).T
    return (lon + 180) % 360 - 180, lat


fig, ax = plt.subplots(figsize=(14, 6), subplot_kw={"projection": ccrs.PlateCarree()})
ax.scatter(*lonlat("obs"), s=2, c="tab:green", label=f"obs nodes ({graph['obs'].num_nodes})")
ax.scatter(*lonlat("data"), s=6, c="grey", label=f"data nodes ({graph['data'].num_nodes})")
ax.scatter(*lonlat("hidden"), s=14, c="tab:red", label=f"hidden nodes ({graph['hidden'].num_nodes})")
# the decoder edges of a few obs nodes
src, dst = graph[("hidden", "to", "obs")].edge_index.cpu().numpy()
hlon, hlat = lonlat("hidden")
olon, olat = lonlat("obs")
for k in np.where((olat[dst] > 45) & (olat[dst] < 47) & (olon[dst] > 0) & (olon[dst] < 10))[0]:
    ax.plot([hlon[src[k]], olon[dst[k]]], [hlat[src[k]], olat[dst[k]]], c="k", lw=0.8)
ax.coastlines()
ax.set_extent([-10, 25, 35, 60])
ax.legend(loc="lower left")

**Question:** with real observations, the decoder would still predict at all 40,320 obs nodes, but the loss would only see the few hundred observed ones of each sample. Is it a problem that the graph has nodes where there is never an observation (e.g. the oceans)?

## 4. The model: a new decoder

The model section of setup 1 adds a decoder group `obs`:

In [ ]:
print(OmegaConf.to_yaml({"decoders": {"obs": {k: v for k, v in finetune.model.decoders.obs.items() if k != "mapper"}}}))
print("trainable parameters per edge of the obs decoder:", finetune.model.decoders.obs.mapper.trainable_size)
print("trainable parameters per obs node:", finetune.model.node_trainable_parameters.obs)

Two choices matter here:

- **`target_node_features`**: what the decoder knows about each point it predicts, besides the hidden state. The ERA5 decoder uses `encoded_data`, the state of the point after the encoder: there is none for `obs`, which has no encoder. Instead, we give the decoder the **coordinates** and the **forcings** of the point (time of day, insolation, land-sea mask, orography).
- **No trainable parameters per node or edge** (`trainable_size: 0`, `node_trainable_parameters.obs: 0`). With them, the decoder would learn something specific to every location. Without them, it is a function of the hidden state and of the properties of the point only: it can decode **anywhere**, also at locations it never saw, e.g. at new observation sites.

## 5. Fine-tune

### 5.0 The pretrained model

The pretrained model is the deterministic model of session 5, trained with anemoi-core `main` for 2000 steps on 2010-2020 (about 8 minutes on an A100). If the checkpoint of `PRETRAINED_RUN_ID` does not exist, the cell below trains it and uses the new run.

In [ ]:
client = mlflow.tracking.MlflowClient(tracking_uri=str(OUTPUT_ROOT / "logs" / "mlflow"))
if not PRETRAINED.exists():
    print(f"{PRETRAINED} does not exist: training configs/pretrain_o48.yaml")
    run_training("pretrain_o48", OVERRIDES)
    PRETRAINED_RUN_ID = latest_run(client, "pretrain-o48").info.run_id
    PRETRAINED = OUTPUT_ROOT / "checkpoint" / PRETRAINED_RUN_ID / "last.ckpt"
    FINETUNE = [f"training.checkpoint.source.path={PRETRAINED}"]
print(PRETRAINED)

### 5.1 Starting from a checkpoint: `training.checkpoint`

On `main`, how a training starts from a checkpoint is set in `training.checkpoint`, in three stages:

| Stage | Config group | Here |
|---|---|---|
| `source`: where the checkpoint is | `training/checkpoint/source`: `local`, `run` (an earlier run id), `s3`, `http` | `local`: the file `PRETRAINED`. With `run` we could give `PRETRAINED_RUN_ID` instead. |
| `loading`: what to take from it | `training/checkpoint/loading`: `warm_start` (resume the same run), `weights_only`, `cold_start`, `transfer_learning` | `transfer_learning`: every weight that exists in both models with the same shape. Our new decoder does not exist in the checkpoint, so it starts from random weights. |
| `modifiers`: what to do with the model then | `training/checkpoint/modifiers`: `freezing` | freeze everything that was pretrained |

In [ ]:
print(OmegaConf.to_yaml(finetune.training.checkpoint))

This is the idea of setup 1: **freeze the rest of the model and train only the new obs decoder**, on the dates that both datasets have (the pretrained model needs ERA5 O48 as input, the obs decoder needs `obs` as its target). With everything pretrained frozen, the forecast of the model stays exactly as it was: the new decoder learns to read the quantities of `obs` from the hidden state of the pretrained model.

**Question:** we also freeze `model.model.node_attributes` and the `*_graph_provider` modules. What are they? (Hint: `model.node_trainable_parameters` and `model.edge_trainable_parameters` in the config.)

### 5.2 Train

1000 steps take about **5 minutes** on the course GPU (on an A100). In the output, look for the lines of the checkpoint pipeline: how many weights were loaded, which modules were frozen, and that `obs` was initialised randomly. In the model summary, compare the trainable and the non-trainable parameters.

In [ ]:
run_training("finetune_obs_decoder", OVERRIDES + DATES + FINETUNE)

## 6. Results

### 6.1 Loss curves

In [ ]:
run = latest_run(client, "finetune-obs-decoder")
print(f"run {run.info.run_name} ({run.info.run_id}), status {run.info.status}")
print(sorted(m for m in run.data.metrics if m.startswith("val_")))

In [ ]:
plot_metrics(client, [run], ["train_multi_dataset_loss_epoch", "val_obs_lossvariablemapper_loss", "val_data_lossvariablemapper_loss"])

**What to look for**
- The **obs** validation loss falls: the new decoder learns.
- The **data** (ERA5 O48) validation loss is flat: the frozen model does not change.

### 6.2 The predictions against `obs`

We run the fine-tuned model ourselves, on a date of the validation period, like a forecast: load the **inference checkpoint**, give it the inputs (ERA5 O48 at *t*-6h and *t*), and compare the predicted `obs` at *t*+6h with the truth.

In [ ]:
checkpoint = OUTPUT_ROOT / "checkpoint" / run.info.run_id / "inference-last.ckpt"
model = torch.load(checkpoint, map_location="cuda", weights_only=False).cuda().eval()
print("datasets:", list(model.data_indices))

# the datasets, opened exactly as in the training (dataloader.validation.datasets.<name>.dataset_config)
datasets = open_datasets({name: cfg.dataset_config for name, cfg in finetune.dataloader.validation.datasets.items()})

# a forecast from the second date of the validation period (the first one is the input at t-6h)
forecast_date = str(datasets["obs"].dates[1])
prediction = predict(model, datasets, forecast_date)
truth = target(datasets, "obs", forecast_date)
print("forecast from", forecast_date)
print({name: list(p)[:6] for name, p in prediction.items()})

The ERA5 decoder `"0"` also predicts `t_850` and `z_500`, on the O48 grid. As a **baseline** for the obs decoder, take the ERA5 forecast at the O48 point nearest to each O96 point. The other baseline is the **climatology**: the mean of each variable.

In [ ]:
from scipy.spatial import cKDTree


def xyz(lat, lon):
    lat, lon = np.deg2rad(lat), np.deg2rad(lon)
    return np.stack([np.cos(lat) * np.cos(lon), np.cos(lat) * np.sin(lon), np.sin(lat)], axis=-1)


nearest_o48 = cKDTree(xyz(datasets["data"].latitudes, datasets["data"].longitudes)).query(
    xyz(datasets["obs"].latitudes, datasets["obs"].longitudes))[1]
# area weights: O96 points are not all the same size
weights = np.cos(np.deg2rad(datasets["obs"].latitudes))


def rmse(a, b):
    return np.sqrt(np.average((a - b) ** 2, weights=weights))


rows = {}
for v in OBSERVED:
    row = {"obs decoder": rmse(prediction["obs"][v], truth[v]),
           "climatology": rmse(datasets["obs"].statistics["mean"][datasets["obs"].name_to_index[v]], truth[v])}
    if v in prediction["data"]:
        row["ERA5 decoder (nearest O48 point)"] = rmse(prediction["data"][v][nearest_o48], truth[v])
    rows[v] = row
pd.DataFrame(rows).T

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
sample = np.random.default_rng(0).choice(len(weights), 3000, replace=False)  # not all 40,320 points
for ax, v in zip(axes, ["t_850", "z_500", "u_500"]):
    ax.scatter(truth[v][sample], prediction["obs"][v][sample], s=4, label="obs decoder")
    if v in prediction["data"]:
        ax.scatter(truth[v][sample], prediction["data"][v][nearest_o48][sample], s=4, alpha=0.5, label="ERA5 decoder")
    lims = [truth[v].min(), truth[v].max()]
    ax.plot(lims, lims, "k--", lw=1)
    ax.set(title=v, xlabel="truth (obs)", ylabel="predicted")
    ax.legend()
plt.tight_layout()

**Questions**
- How does the obs decoder compare with climatology? With the ERA5 decoder, after only 1000 steps?
- The pretrained model never saw `u` or `v`! Where can the obs decoder get the wind from? (Hint: geostrophic balance.)
- Here the truth is a smooth, complete field. Real observations measure at a point, and have errors. Which RMSE would a perfect model get against them?

### 6.3 Decoding anywhere

The obs decoder predicts at every node of the obs grid. Here is the predicted wind at 500 hPa, a quantity the pretrained model never had, and its error:

In [ ]:
lat, lon = datasets["obs"].latitudes, datasets["obs"].longitudes
fig, axes = plt.subplots(2, 2, figsize=(16, 9), subplot_kw={"projection": ccrs.PlateCarree()})
for row, v in zip(axes, ["u_500", "t_850"]):
    vmin, vmax = np.percentile(truth[v], [1, 99])
    cmap = "RdBu_r" if v == "u_500" else "RdYlBu_r"
    im = scatter_map(row[0], lat, lon, prediction["obs"][v], title=f"{v}: obs decoder", s=1, cmap=cmap, vmin=vmin, vmax=vmax)
    plt.colorbar(im, ax=row[0], shrink=0.8)
    error = prediction["obs"][v] - truth[v]
    limit = np.percentile(np.abs(error), 99)
    im = scatter_map(row[1], lat, lon, error, title=f"{v}: obs decoder - truth", s=1, cmap="bwr", vmin=-limit, vmax=limit)
    plt.colorbar(im, ax=row[1], shrink=0.8)

**Question:** where are the errors largest? Why could that be?

On `main`, the places the decoder can predict at are the nodes of the obs dataset, fixed when the graph is built. Because the obs decoder has no trainable parameters per node or edge, the same weights would work with other nodes: decoding at **arbitrary locations** only needs a graph to them. With richer batches (section 8), the target locations are given with each sample, and the edges are built on the fly.

## 7. Encode observations (setup 2)

Now `obs` is an **input**: a new encoder `obs` maps it to the hidden mesh, where its result is added to that of the ERA5 encoder (`model.latent_aggregator: SumAggregator`). The model gets `obs` at *t*-6h and *t*: here the same reanalysis, but at a higher resolution and with variables it did not have (wind, humidity). All variables of `obs` are forcings, so `obs` is an *input-only* dataset: no decoder, no loss.

In [ ]:
encode = load_config("encode_obs", OVERRIDES + DATES + FINETUNE)
pd.DataFrame(config_diff(pretrain, encode), columns=["setting", "pretrain_o48", "encode_obs"])

**Questions**
- In this setup the whole model is trained (no freezing), with a smaller learning rate. Why not freeze the pretrained model here?
- With real observations, the gaps would be filled in with the mean. How could the model tell a missing observation from one that is equal to the mean? What could we add to the dataset to help it?
- ERA5 is itself an analysis made with the observations. What does this mean for what a model can gain from them? What would be a more realistic setup for data assimilation?

To see if `obs` helps, compare with a model trained in the same way **without** it: the pretrained model, trained for 1000 more steps on the same dates. The two trainings take about 10 minutes together; you can also start them in a terminal (see `README.md`).

In [ ]:
DATA_DATES = [d.replace(".obs.", ".data.") for d in DATES]  # the same periods, for a config without `obs`
run_training("encode_obs", OVERRIDES + DATES + FINETUNE)
run_training("pretrain_o48", OVERRIDES + DATA_DATES + FINETUNE + [
    "training/checkpoint/source=local", "training/checkpoint/loading=transfer_learning",
    "training.max_steps=1000", "training.optimization.lr=1e-4", "training.optimization.lr_scheduler.warmup_t=50",
    "dataloader.batch_size.validation=1", "dataloader.limit_batches.validation=null",
    "diagnostics.log.mlflow.run_name=no-obs-control"])

In [ ]:
runs = [latest_run(client, "encode-obs"), latest_run(client, "no-obs-control")]
plot_metrics(client, runs, ["val_data_lossvariablemapper_loss", "val_data_mse_metric/data/z_500/1", "val_data_mse_metric/data/t_850/1"])

**Question:** does the model with `obs` as input forecast ERA5 O48 better? By how much, and is that surprising, given what `obs` is here?

## 8. Looking ahead: richer batches and dynamic observations

The next release of anemoi-core (branch `feat/richer-batch`, to be merged in the coming days) changes how data flows through anemoi, with observations in mind. What changes for you:

**Batches carry more than tensors.** On `main` a batch is a dictionary `{dataset name: tensor}`. With richer batches it is a `Batch` of `Source`s: each source has its `values` together with its `coordinates`, `variables`, `timedeltas` (the time of each value relative to the reference time) and statistics. Code that indexes the batch (custom callbacks, losses, tasks) uses `batch["obs"].values`.

**Tabular observations.** A dataset whose `dataset_config` has a `window` is read as a **tabular** dataset (`TabularDataReader`): each sample holds only the observations made in the time window, each with its own position and time. No binning onto a grid, no NaN gaps, no imputer, no mask:

```yaml
dataloader:
  training:
    datasets:
      surface_obs:
        dataset_config:
          dataset: ${system.input.surface_obs}
          frequency: ${data.frequency}
          window: (-6h,0]          # NEW: all observations of the 6 hours up to each time
          drop: [...]
```

**Graphs built on the fly.** The positions of tabular observations change from sample to sample, so the edges to and from them are built for every batch, with the same edge builders as before (`KNNEdges`, `CutOffEdges`, `ReversedKNNEdges`). The graph section is unchanged: whether edges are static or dynamic follows from the type of the dataset. These edges cannot have trainable parameters (`trainable_size: 0`, as we already used). New attributes `Timedeltas` give the nodes and edges the time of each observation.

**Decoding anywhere at inference.** `predict_step` takes a `target_template`: positions (and times) to decode at, without data. Setup 1 is then a model that predicts the observed quantities at any location.

**Config changes** (among others):

| `main` | richer batches |
|---|---|
| `dataset_fusing_strategy: "not_supported"` | `"none"` or `"sequential"` (several datasets share one encoder) + `node_embedder` |
| decoder target features: `coordinates`, `forcings`, ... | also `target_forcings`: forcings at the *output* time and location (e.g. the satellite viewing angles of an observation) |
| `NaNMaskScaler` (`nan_mask_weights`), imputer for gaps in targets | removed: use `ignore_nans: True` on the loss |
| any DDP strategy | with tabular datasets: `static_graph: false`, `find_unused_parameters: true` |

A full example with ERA5 and seven observation types (encoded and decoded) is in `training/tests/config/test_single.yaml` of anemoi-core (branch `feat/richer-batch`).

**From this notebook to dynamic observations**

| | `main` (this notebook) | richer batches |
|---|---|---|
| observation dataset | gridded (here complete ERA5 O96 fields; real observations binned onto a grid, NaN where there is no observation) | the observations themselves, with `window` |
| dates | same dates as ERA5 | observations from a time window around each ERA5 date |
| obs nodes | fixed: the grid points | the observations of each sample |
| gaps | none here; real gridded observations need an imputer + `nan_mask_weights` | none |
| setup 1: decode | `decoders.obs`, `target_node_features: [coordinates, forcings]` | the same, possibly with `target_forcings` |
| setup 2: encode | `encoders.obs`, all observed variables as forcings | the same, `trainable_size: 0` |

## 9. Tasks

### Task 1: fine-tune more
Fine-tune the whole model instead of the obs decoder only, with a smaller learning rate:
```
run_training("finetune_obs_decoder", OVERRIDES + DATES + FINETUNE + ["training.checkpoint.modifiers.0.submodules_to_freeze=[]",
             "training.optimization.lr=1e-4", "diagnostics.log.mlflow.run_name=finetune-all"])
```
Does the obs decoder get better? What happens to the ERA5 validation loss?

### Task 2: what does the decoder need to know?
Change `model.decoders.obs.target_node_features`, e.g. to `["coordinates"]` only, or add `"trainable_parameters"` (and set `model.node_trainable_parameters.obs: 8`). How do the results change? Which of these models could decode at new locations?

### Task 3: other variables
Add more variables of the O96 dataset to `obs` (other levels, `2t`, `10u`, ...): copy `configs/obs/era5_o96.yaml`, change the variable lists and `select`, and select it with the override `obs=<your file name>`. Which variables does the obs decoder predict well, which not?

### Task 4: encode and decode
Combine setups 1 and 2: `obs` at *t*-6h and *t* as input, and `obs` at *t*+6h as target. Make the observed variables prognostic (`data.datasets.obs.diagnostic: []`), and give `obs` both an encoder and a decoder, a residual entry and a loss.